# Stage 5 — Distillation

Compress the recipe. The R1 paper shows that distilling a big reasoning model into a small one works *better* than RL'ing the small one directly.

We do the simplest possible version: a smaller **student** model is trained to match the **teacher's** output distribution, using KL-divergence on logits.

Teacher = the model from notebook 05. Student = a 1-layer half-width version.

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F, random, re
torch.manual_seed(0); random.seed(0)

VOCAB = list('0123456789+=TA .')
stoi = {c: i for i, c in enumerate(VOCAB)}; itos = {i: c for c, i in stoi.items()}
V, PAD, EOS, BLOCK = len(VOCAB), stoi[' '], stoi['.'], 24
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)

class LM(nn.Module):
    def __init__(self, V, d=64, h=4, L=2, block=BLOCK):
        super().__init__()
        self.tok = nn.Embedding(V, d); self.pos = nn.Embedding(block, d)
        self.blocks = nn.ModuleList([nn.TransformerEncoderLayer(d, h, dim_feedforward=4*d, batch_first=True, activation='gelu') for _ in range(L)])
        self.head = nn.Linear(d, V)
    def forward(self, x):
        T = x.shape[1]
        h = self.tok(x) + self.pos(torch.arange(T, device=x.device))
        mask = nn.Transformer.generate_square_subsequent_mask(T).to(x.device)
        for blk in self.blocks: h = blk(h, src_mask=mask)
        return self.head(h)

teacher = LM(V, d=64, L=2)
teacher.load_state_dict(torch.load('after_reject_sft.pt'))
teacher.eval()
student = LM(V, d=32, L=1)   # half width, half depth
print('teacher params:', sum(p.numel() for p in teacher.parameters())/1e3, 'K')
print('student params:', sum(p.numel() for p in student.parameters())/1e3, 'K')

teacher params: 103.568 K
student params: 14.512 K


In [2]:
# Build a distillation set: teacher's own greedy completions on every prompt
@torch.no_grad()
def teacher_complete(prompt, max_new=16):
    x = torch.tensor([encode(prompt)])
    for _ in range(max_new):
        nxt = torch.argmax(teacher(x[:, -BLOCK:])[:, -1, :], -1, keepdim=True)
        x = torch.cat([x, nxt], 1)
        if nxt.item() == EOS: break
    return decode(x[0].tolist())

distill_set = [teacher_complete(f'{a}+{b}=') for a in range(10) for b in range(10)]
print('distill set size:', len(distill_set))
for ex in distill_set[:5]: print(' ', ex)

distill set size: 100
  0+0=T0+18A0.
  0+1=T0+1=A1.
  0+2=T0+21A2.
  0+3=T0+33A4.
  0+4=T0+18A4.


In [3]:
def pad_batch(strings):
    ids = [encode(s) for s in strings]
    L = max(len(x) for x in ids)
    return torch.tensor([x + [PAD]*(L-len(x)) for x in ids])

opt = torch.optim.AdamW(student.parameters(), lr=3e-3)
T_temp = 2.0
for step in range(500):
    batch = random.choices(distill_set, k=16)
    x = pad_batch(batch)
    with torch.no_grad():
        t_logits = teacher(x[:, :-1]) / T_temp
    s_logits = student(x[:, :-1]) / T_temp
    loss = F.kl_div(F.log_softmax(s_logits, -1), F.softmax(t_logits, -1), reduction='batchmean') * (T_temp**2)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0: print(f'step {step:3d}  KL {loss.item():.3f}')

step   0  KL 104.564


step 100  KL 21.583


step 200  KL 9.004


step 300  KL 5.923


step 400  KL 5.039


In [4]:
@torch.no_grad()
def greedy(model, prompt, max_new=16):
    x = torch.tensor([encode(prompt)])
    for _ in range(max_new):
        nxt = torch.argmax(model(x[:, -BLOCK:])[:, -1, :], -1, keepdim=True)
        x = torch.cat([x, nxt], 1)
        if nxt.item() == EOS: break
    return decode(x[0].tolist())

def pass_rate(model, n=100):
    c = 0
    for _ in range(n):
        a, b = random.randint(0,9), random.randint(0,9)
        m = re.search(r'A(\d+)\.', greedy(model, f'{a}+{b}='))
        if m and int(m.group(1)) == a + b: c += 1
    return c / n

print(f'teacher pass-rate: {pass_rate(teacher):.0%}')
print(f'student pass-rate: {pass_rate(student):.0%}')

teacher pass-rate: 36%


student pass-rate: 29%


## Takeaway

A model with **half the layers and half the width** picked up most of the teacher's skill via plain KL on logits.

This is exactly why the R1 distilled checkpoints (Qwen-1.5B, 7B, 14B) punch so far above their weight: the teacher already did all the hard exploration via RL.

## Course recap

| Stage | Notebook | What it adds |
|---|---|---|
| Pre-train | 02 | Language |
| Cold-start SFT | 03 | Format |
| RL (GRPO) | 04 | Skill |
| Reject-SFT | 05 | Stability |
| Distill | 06 | Efficiency |

That is the whole DeepSeek R1 recipe.